In [2]:
pip install ultralytics --upgrade


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.0 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.9 MB/s eta 0:00:00:00:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 20.2 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 2.3 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.8 MB/s eta 0:00:000:00:010:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 65.4 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.9.41
    Uninstalling nvidia-nvjitlink-cu12-12.9.41:
      Successfully uninstalled nvidia-nvjitlink-cu12-12.9.41
  Attempting uninstall: nvidia-curand-cu12
    Found exi

In [1]:
import torch
import torch.nn as nn
import os
import shutil
from pathlib import Path
import json
import cv2
import numpy as np
from sklearn.model_selection import train_test_split
import yaml
from ultralytics import YOLO
from torch.utils.data import DataLoader
from torchvision.datasets import VisionDataset
from torchvision.transforms import ToTensor
from tqdm import tqdm
from collections import deque
import albumentations as A
from torch.utils.data import Dataset

/usr/local/lib/python3.11/dist-packages/albumentations/__init__.py:28: UserWarning: A new version of Albumentations is available: '2.0.8' (you have '2.0.5'). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


In [11]:

class YoloOrganizer:

    def __init__(self, images_dir, labels_dir, output_dir):
        self.images_dir = Path(images_dir)
        self.labels_dir = Path(labels_dir)
        self.output_dir = Path(output_dir)

        self._validate_paths()
        self._create_output_dirs()


    def _validate_paths(self):
        if not self.images_dir.exists():
            raise FileNotFoundError(f"[ERROR] Images directory not found: {self.images_dir}")

        if not self.labels_dir.exists():
            raise FileNotFoundError(f"[ERROR] Labels directory not found: {self.labels_dir}")

    def _create_output_dirs(self):
        for split in ["train", "val"]:
            (self.output_dir / split / "images").mkdir(parents=True, exist_ok=True)
            (self.output_dir / split / "labels").mkdir(parents=True, exist_ok=True)


    def convert_bbox(self, bbox, img_w, img_h, format_from="xyxy"):


        try:
            if format_from == "xyxy":
                x1, y1, x2, y2 = bbox
                cx = ((x1 + x2) / 2) / img_w
                cy = ((y1 + y2) / 2) / img_h
                w = (x2 - x1) / img_w
                h = (y2 - y1) / img_h

            elif format_from == "xywh":
                x, y, w_box, h_box = bbox
                cx = (x + w_box / 2) / img_w
                cy = (y + h_box / 2) / img_h
                w = w_box / img_w
                h = h_box / img_h

            else:
                raise ValueError(f"[ERROR] Unsupported bbox format: {format_from}")

            return cx, cy, w, h

        except Exception as e:
            raise ValueError(f"[ERROR] Failed to convert bbox {bbox}: {e}")



    def write_yolo_label(self, annotations, save_path):
        """
        annotations: list of (class_id, cx, cy, w, h)
        """
        if not annotations:
            print(f"[WARNING] No annotations for {save_path.name}")
            return

        with open(save_path, "w") as f:
            for ann in annotations:
                if len(ann) != 5:
                    raise ValueError(f"[ERROR] Invalid annotation format: {ann}")
                cls, cx, cy, w, h = ann
                f.write(f"{cls} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n")

  

    def split_dataset(self, train_ratio=0.8):
        if not 0 < train_ratio < 1:
            raise ValueError("[ERROR] train_ratio must be between 0 and 1")

        images = list(self.images_dir.glob("*.jpg")) + \
                 list(self.images_dir.glob("*.png")) + \
                 list(self.images_dir.glob("*.jpeg"))

        if not images:
            raise RuntimeError("[ERROR] No images found in images directory")

        valid_images = []
        for img in images:
            label = self.labels_dir / f"{img.stem}.txt"
            if label.exists():
                valid_images.append(img)
            else:
                print(f"[WARNING] Missing label for image: {img.name}")

        if not valid_images:
            raise RuntimeError("[ERROR] No valid image-label pairs found")

        train_imgs, val_imgs = train_test_split(
            valid_images,
            test_size=1 - train_ratio,
            random_state=42
        )

        self._copy_pairs(train_imgs, "train")
        self._copy_pairs(val_imgs, "val")

        print(f"[INFO] Dataset split complete → Train: {len(train_imgs)}, Val: {len(val_imgs)}")
        return len(train_imgs), len(val_imgs)

    def _copy_pairs(self, images, split):
        for img in images:
            shutil.copy2(img, self.output_dir / split / "images" / img.name)
            shutil.copy2(
                self.labels_dir / f"{img.stem}.txt",
                self.output_dir / split / "labels" / f"{img.stem}.txt"
            )



    def create_dataset_yaml(self, yaml_path, class_names):
        if not class_names or not isinstance(class_names, list):
            raise ValueError("[ERROR] class_names must be a non-empty list")

        data = {
            "path": str(self.output_dir.resolve()),
            "train": "train/images",
            "val": "val/images",
            "nc": len(class_names),
            "names": class_names
        }

        with open(yaml_path, "w") as f:
            yaml.dump(data, f)

        print(f"[INFO] dataset.yaml created at: {yaml_path}")


In [12]:
class DataAugmentationPipeline:
    def __init__(self):
        import albumentations as A

        self.transform = A.Compose([
            A.HorizontalFlip(p=0.5),

            A.ShiftScaleRotate(
                shift_limit=0.05,
                scale_limit=0.15,
                rotate_limit=10,
                p=0.5
            ),

            A.RandomBrightnessContrast(p=0.3),
            A.HueSaturationValue(p=0.3),
            A.CLAHE(p=0.2),

            A.OneOf([
                A.MotionBlur(3),
                A.GaussianBlur(3),
                A.MedianBlur(3),
            ], p=0.2),

            A.OneOf([
                A.GaussNoise(var_limit=(10, 40)),
                A.ISONoise(),
            ], p=0.2),

            A.OneOf([
                A.RandomShadow(),
                A.RandomSunFlare(src_radius=100),
            ], p=0.15),

            A.Resize(640, 640),
        ],
        bbox_params=A.BboxParams(
            format='yolo',
            label_fields=['class_labels'],
            min_visibility=0.2
        ))


In [13]:
class AlbumentationsDataset(Dataset):
    def __init__(self, yaml_path, imgsz=320, transform=None, mode="train"):
        import yaml
        with open(yaml_path, 'r') as f:
            cfg = yaml.safe_load(f)

        self.imgsz = imgsz
        self.transform = transform
        self.mode = mode

        base = Path(cfg["path"])
        img_dir = base / mode / "images"
        label_dir = base / mode / "labels"

        self.images = sorted(list(img_dir.glob("*.jpg")) +
                             list(img_dir.glob("*.png")) +
                             list(img_dir.glob("*.jpeg")))

        self.labels = [label_dir / f"{img.stem}.txt" for img in self.images]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        img_path = str(self.images[index])
        label_path = str(self.labels[index])

        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        h, w = img.shape[:2]

        # Load YOLO labels
        boxes = []
        if os.path.exists(label_path):
            with open(label_path, "r") as f:
                for line in f.readlines():
                    cls, cx, cy, bw, bh = map(float, line.strip().split())
                    boxes.append([cx, cy, bw, bh, int(cls)])

        boxes = np.array(boxes)
        if len(boxes) == 0:
            boxes = np.zeros((0, 5))

        bboxes = boxes[:, :4]
        class_labels = boxes[:, 4].astype(int)

        # Albumentations transform
        if self.transform:
            tr = self.transform(image=img, bboxes=bboxes, class_labels=class_labels)
            img = tr["image"]
            bboxes = tr["bboxes"]
            class_labels = tr["class_labels"]

        # Convert to tensor
        img = img.transpose(2, 0, 1)  # HWC → CHW
        img = img.astype(np.float32) / 255.0

        labels = []
        for cls, bb in zip(class_labels, bboxes):
            labels.append([cls] + list(bb))  # YOLO format

        labels = np.array(labels, dtype=np.float32)

        return img, labels

In [27]:

organizer = YoloOrganizer(
    images_dir="/kaggle/input/finalaggregatedataset/FinalAggregateDataset/Images",
    labels_dir="/kaggle/input/finalaggregatedataset/FinalAggregateDataset/Labels",
    output_dir="/kaggle/working/yolo_dataset"
)

organizer.split_dataset(train_ratio=0.8)

organizer.create_dataset_yaml(
    yaml_path="/kaggle/working/dataset.yaml",
    class_names=["cheating", "not_cheating"]
)

aug = DataAugmentationPipeline().transform


def custom_train_dataset(trainer):
    trainer.train_dataset = AlbumentationsDataset(
        yaml_path="/kaggle/working/dataset.yaml",
        imgsz=640,
        transform=aug,
        mode="train"
    )
model = YOLO('yolov8m.pt')  

model.add_callback("on_pretrain_routine_start", custom_train_dataset)

[WARNING] Missing label for image: IMG-20251210-WA0078.jpg
[WARNING] Missing label for image: IMG_20250617_143326.jpg
[WARNING] Missing label for image: IMG_20260108_112037.jpg
[WARNING] Missing label for image: IMG-20251210-WA0077.jpg
[WARNING] Missing label for image: IMG_20250617_143619_1.jpg
[WARNING] Missing label for image: IMG-20251210-WA0076.jpg
[WARNING] Missing label for image: IMG_20250617_143118_1.jpg
[WARNING] Missing label for image: IMG_20250617_143658_1.jpg
[WARNING] Missing label for image: IMG_20250617_142807_1.jpg
[WARNING] Missing label for image: IMG-20251210-WA0079.jpg
[WARNING] Missing label for image: IMG_20260108_112034.jpg
[INFO] Dataset split complete → Train: 549, Val: 138
[INFO] dataset.yaml created at: /kaggle/working/dataset.yaml


/usr/local/lib/python3.11/dist-packages/albumentations/core/validation.py:87: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
/tmp/ipykernel_110/2876183458.py:26: UserWarning: Argument(s) 'var_limit' are not valid for transform GaussNoise
  A.GaussNoise(var_limit=(10, 40)),


In [28]:
results = model.train(
    data="/kaggle/working/dataset.yaml",
    epochs=100,
    imgsz=640,
    batch=8,

 
    patience=15,

    
    lr0=0.01,
    lrf=0.01,
    momentum=0.937,

    
    augment=False,
    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.0,
    degrees=0.0,
    translate=0.0,
    scale=0.0,
    fliplr=0.0,
    mosaic=0.0,
    mixup=0.0,
    copy_paste=0.0,

    # Logging & diagnostics
    plots=True,
    save=True,
    save_period=5,
    verbose=True
)

Ultralytics 8.3.252 🚀 Python-3.11.11 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/dataset.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=0.0, multi_scale=False, name=train9, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=15, perspective=0.0, plots=True, pose=12.

/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


                   all        185        503      0.878      0.742      0.852      0.745
              cheating        156        263      0.876      0.696       0.83      0.714
          not_cheating        127        240      0.881      0.787      0.874      0.776
Speed: 0.2ms preprocess, 8.5ms inference, 0.1ms loss, 2.9ms postprocess per image
Results saved to /kaggle/working/runs/detect/train9


In [29]:


folder_path = "/kaggle/working/runs/detect/train9"


output_zip = "/kaggle/working/Aggregate640_mModel"

shutil.make_archive(output_zip, 'zip', folder_path)

print("Zipping complete! The file is saved as:", output_zip + ".zip")


Zipping complete! The file is saved as: /kaggle/working/Aggregate640_mModel.zip


In [ ]:

# def predict_and_zip(input_dir, model_path="runs/detect/cheating_detection_Correcto/weights/best.pt", output_dir="outputs", zip_name="outputs.zip"):
#     # Clean up previous outputs
#     if os.path.exists(output_dir):
#         shutil.rmtree(output_dir)
#     os.makedirs(output_dir, exist_ok=True)

#     # Load YOLOv8 model (class names will be loaded from the .pt file)
#     model = YOLO(model_path)

#     # Predict on input directory
#     results = model.predict(
#         source=input_dir,
#         save=True,         # Save images with bounding boxes
#         save_txt=True,     # Save label files
#         project=output_dir,
#         name="predict",
#         exist_ok=True,
#         imgsz=640,
#         conf=0.25
#     )

#     # Zip up the output images + labels
#     prediction_path = os.path.join(output_dir, "predict")
#     with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
#         for root, _, files in os.walk(prediction_path):
#             for file in files:
#                 file_path = os.path.join(root, file)
#                 arcname = os.path.relpath(file_path, prediction_path)
#                 zipf.write(file_path, arcname)

#     print(f"\n✅ Predictions (with class names) saved to {prediction_path}")
#     print(f"📦 Zipped into: {zip_name}")


# predict_and_zip(input_dir="/kaggle/input/corrected-primary-dataset/PRIMARY Dataset/Evaluation")
